# Mixed elasticity and the incompressible limit

The native GaLS method solves the displacement–Herrmann-pressure system
$\sigma=2\mu\varepsilon(u)-pI$, $-\nabla\cdot\sigma=f$ and
$\nabla\cdot u+p/\lambda=0$. Three rigid modes are retained on every macrotriangle.
GaLS uses equal-order P1/P1, P2/P2 or P3/P3 local fields. Taylor–Hood uses
P2/P1 or P3/P2. Set `lame_lambda=np.inf` for the incompressible limit.

The trigonometric exact family below has a bounded force and nonzero limiting
pressure. Its amplitude-one convention agrees with the MSL GaLS elasticity case;
it does not mix the incompatible displacement, pressure and force factors
printed in section 6 of arXiv:2403.16890. Exact data are differentiated and tested
independently in `examples/elasticity_data.py` and `tests/test_elasticity_data.py`.
This notebook executes pyMHM, without an external comparison solver.

The application declares P1/P1 local equations using the shared tabulation and
integration kernels. With $R(u,p)=-\nabla p$ for constant shear and affine
fine-cell displacement, the GaLS form is

$$
\begin{aligned}
a_T((u,p),(v,q))={}&2\mu(\varepsilon(u),\varepsilon(v))_T
 -(p,\operatorname{div}v)_T-(q,\operatorname{div}u)_T\\
&-\lambda^{-1}(p,q)_T-\alpha h_T^2(R(u,p),R(v,q))_T,\\
L_T(v,q)={}&(f,v)_T+\alpha h_T^2(f,R(v,q))_T,\\
B_T\lambda_T={}&\langle\lambda_T,v\rangle_{\partial T},\qquad
C_T(u,p)=-\langle u,\eta\rangle_{\partial T}.
\end{aligned}
$$

Here $\alpha=1/(4\mu)$ is half the sufficient P1 constant-shear bound;
the local inverse constant is one. The skeleton uses P1 traces aligned with
four fine edges per macroface, and all three rigid modes have physical integral
moments about the same declared global center. The global `Equation` supplies
negative displacement boundary moments. Its finite-compliance identity and
incompressible pressure gauge are applied to the reconstructed pressure,
while pure traction uses the three stated displacement integrals.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from functools import partial
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from threadpoolctl import threadpool_limits
from examples.elasticity_data import TrigonometricElasticityData
from examples.plot_elasticity import field_arrays
from examples.plot_mesh import draw_macro_mesh
from examples.formulations.elasticity import (
    DisplacementPressureSpace, displacement_pressure_equations,
    displacement_pressure_fields,
)
from pymhm import Equation, MultiscaleProblem, assemble, TriangleMesh, SkeletonSpace, FaceSpace
from pymhm.fem.scalar.operators import boundary_data

mesh = TriangleMesh.unit_square(2)
skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces), 2)
local_space = DisplacementPressureSpace(mesh, skeleton, 4, 8)
exact = TrigonometricElasticityData(lame_lambda=4999.0)
provider = partial(
    displacement_pressure_equations, space=local_space,
    source=exact.source, lame_lambda=exact.lame_lambda,
)
boundary, _ = boundary_data(skeleton, (0.0, 0.0), order=8)
problem = MultiscaleProblem(
    Equation(0, -np.r_[boundary, np.zeros(3 * len(mesh.cells))]),
    provider, range(len(mesh.cells)), skeleton.size, (3,) * len(mesh.cells),
)
with threadpool_limits(limits=1):
    system = assemble(problem)
    compliance_scale = max(record[6] for record in system.local_metadata)
    pressure_identity = system.mean_constraint(
        [record[5] / compliance_scale for record in system.local_metadata], 0.0,
    )
    coefficients = system.solve(constraints=(pressure_identity,))
    result = displacement_pressure_fields(system, coefficients, local_space, exact.lame_lambda)
print("displacement L2", result.l2_error(exact.displacement, 10))
print("pressure L2", result.pressure_l2_error(exact.pressure, 10))
print("full stress L2", result.stress_l2_error(exact.stress, 10))
print("compressibility L2", result.compressibility_l2(10))
assert result.hybrid.residual < 1e-11
assert all(len(modes) == 3 for modes in result.hybrid.coarse)


## Pressure on the actual macro mesh

Dirichlet displacement is enforced through skeletal moments. Reconstructed local
fields can therefore have jumps at macro interfaces. The following plot retains
each local field separately. Dark contours show the actual macrotriangles; the
error panel displays pointwise pressure error, while the numbers above are
quadrature-integrated norms.

In [ ]:
arrays = field_arrays(result, exact)
triangulation = mtri.Triangulation(*arrays["points"].T, triangles=arrays["cells"])
figure, axes = plt.subplots(1, 3, figsize=(12, 3.4), constrained_layout=True)
fields = (arrays["exact_pressure"], arrays["pressure"],
          np.abs(arrays["pressure"] - arrays["exact_pressure"]))
scale = max(np.max(np.abs(fields[0])), np.max(np.abs(fields[1])))
for index, (axis, values, title) in enumerate(zip(axes, fields, ("Exact pressure", "GaLS P1/P1", "Absolute error"))):
    artist = axis.tripcolor(
        triangulation, values, shading="gouraud", rasterized=True,
        cmap="magma" if index == 2 else "coolwarm",
        vmin=0 if index == 2 else -scale,
        vmax=None if index == 2 else scale,
    )
    draw_macro_mesh(axis, mesh)
    figure.colorbar(artist, ax=axis)
    axis.set(title=title, xlabel="x", ylabel="y", aspect="equal")
plt.show()

## Finite and infinite first Lamé modulus

The pressure mean is fixed by integrated compressibility for finite $\lambda$.
At $\lambda=\infty$, zero mean is a global gauge when the whole boundary has
prescribed displacement. The default trace has degree one, so P1/P1 GaLS requires
at least four fine intervals per trace segment. A P1/P1 local refinement of two
is rejected rather than repaired by a diagonal perturbation.

In [ ]:
fields = []
with threadpool_limits(limits=1):
    for modulus in (1e4, 1e8, np.inf):
        data = TrigonometricElasticityData(modulus)
        provider = partial(
            displacement_pressure_equations, space=local_space,
            source=data.source, lame_lambda=modulus,
        )
        problem = MultiscaleProblem(
            Equation(0, -np.r_[boundary, np.zeros(3 * len(mesh.cells))]),
            provider, range(len(mesh.cells)), skeleton.size, (3,) * len(mesh.cells),
        )
        system = assemble(problem)
        scale = max(record[6] for record in system.local_metadata)
        weights = [
            record[2] if scale == 0 else record[5] / scale
            for record in system.local_metadata
        ]
        pressure_identity = system.mean_constraint(weights, 0.0)
        coefficients = system.solve(constraints=(pressure_identity,))
        numerical = displacement_pressure_fields(system, coefficients, local_space, modulus)
        fields.append(numerical)
        print(modulus, numerical.l2_error(data.displacement, 10),
              numerical.pressure_l2_error(data.pressure, 10))
        assert numerical.hybrid.residual < 1e-11
far = sum(np.linalg.norm(a-b) for a,b in zip(fields[0].values, fields[-1].values))
near = sum(np.linalg.norm(a-b) for a,b in zip(fields[1].values, fields[-1].values))
assert near < 2e-4 * far


## Six refinement levels

The archived series keeps 32 macrotriangles fixed. Each macroface is split into
$s=1,2,3,4,6,8$ segments. P1/P1 uses local refinement $4s$ and P2/P2 uses $2s$,
so both satisfy the trace/local compatibility condition. These are measured
errors against the exact family, not digitized paper points or a matching MSL run.

The full gallery contains bounded-material sweeps, full Cauchy-stress errors,
pressure maps and one-sided interface profiles. Regenerate it with
`pixi run -e notebooks python examples/plot_elasticity.py --workers 4`.
The optional `--reuse-results` argument redraws the stored data without solves.

In [ ]:
import json
from IPython.display import SVG, display

report = json.loads((root / "examples/results/elasticity.json").read_text())
for method in ("gals-p1", "gals-p2"):
    rows = [row for row in report["refinement"] if row["method"] == method]
    assert len(rows) == 6
    print(method, [(row["segments"], row["displacement_l2"], row["pressure_l2"]) for row in rows])
display(SVG(filename=str(root / "docs/figures/elasticity/refinement.svg")))

## Traction data and rigid-motion gauges

`neumann` accepts physical outward Cauchy traction $\sigma n$ on selected boundary
faces. For a pure traction problem, `rigid_moments` fixes the two displacement
integrals and the rotation moment about the mean macro-vertex position. An
unbalanced force or moment is rejected. The mixed stress–displacement–rotation
H(div) method is a separate formulation; this displacement–pressure method does
not provide an equilibrated H(div)-conforming stress reconstruction.

In [ ]:
# Physical Cauchy traction is zero, hence its negative multiplier is also zero.
traction = {int(face): (0.0, 0.0) for face in mesh.boundary_faces}
traction_space = DisplacementPressureSpace(mesh, skeleton, 4, 6)
provider = partial(
    displacement_pressure_equations, space=traction_space,
    source=(0.0, 0.0), lame_lambda=np.inf,
)
traction_boundary, fixed = boundary_data(skeleton, (0.0, 0.0), traction, order=6)
problem = MultiscaleProblem(
    Equation(0, -np.r_[traction_boundary, np.zeros(3 * len(mesh.cells))]),
    provider, range(len(mesh.cells)), skeleton.size, (3,) * len(mesh.cells), fixed=fixed,
)
with threadpool_limits(limits=1):
    system = assemble(problem)
    gauges = tuple(
        system.mean_constraint([record[4][:, i] for record in system.local_metadata], value)
        for i, value in enumerate((0.2, -0.1, 0.0))
    )
    coefficients = system.solve(constraints=gauges)
    rigid = displacement_pressure_fields(system, coefficients, traction_space, np.inf)
assert rigid.l2_error((0.2, -0.1)) < 1e-11
assert rigid.pressure_l2_error(0.0) < 1e-10
print("pure-traction rigid patch", rigid.l2_error((0.2, -0.1)))
